# Notebook 07 - Report Export Summary

Scope: build report-ready summary tables and a figure manifest from existing outputs only. This notebook does not train models, recreate models, generate predictions, recompute SHAP, run preprocessing, or modify earlier notebooks.

## 1. Imports

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

MODEL_ORDER = ['Logistic Regression', 'Random Forest', 'XGBoost']
METRIC_COLUMNS = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'PR_AUC']
REPORT_PREFIXES = ('report_summary_', 'report_table_', 'report_figure_manifest')

## 2. Path Setup

In [2]:
def find_refactor_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        if candidate.name == 'churn_pipeline_v2':
            return candidate
        nested = candidate / 'churn_pipeline_v2'
        if nested.exists() and nested.is_dir():
            return nested.resolve()
    raise FileNotFoundError('Could not locate churn_pipeline_v2 from the current working directory.')


CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()
REFACTOR_ROOT = find_refactor_root(CURRENT_WORKING_DIRECTORY)
REPO_ROOT = REFACTOR_ROOT.parent
TARGET_NOTEBOOK_PATH = REFACTOR_ROOT / 'notebooks' / '07_Report_Export_Summary.ipynb'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from churn_pipeline_v2.src.path import DATA_REPORTS_DIR, FIGURES_DIR

DATA_REPORTS_DIR = Path(DATA_REPORTS_DIR)
FIGURES_DIR = Path(FIGURES_DIR)

path_checks = pd.DataFrame([
    {'Name': 'Current working directory', 'Path': CURRENT_WORKING_DIRECTORY, 'Exists': CURRENT_WORKING_DIRECTORY.exists()},
    {'Name': 'Refactor root', 'Path': REFACTOR_ROOT, 'Exists': REFACTOR_ROOT.exists()},
    {'Name': 'Target notebook', 'Path': TARGET_NOTEBOOK_PATH, 'Exists': TARGET_NOTEBOOK_PATH.exists()},
    {'Name': 'Data reports folder', 'Path': DATA_REPORTS_DIR, 'Exists': DATA_REPORTS_DIR.exists()},
    {'Name': 'Figures folder', 'Path': FIGURES_DIR, 'Exists': FIGURES_DIR.exists()},
])
display(path_checks)
if not DATA_REPORTS_DIR.exists() or not FIGURES_DIR.exists():
    raise FileNotFoundError('Required report output folders are missing.')

,Name,Path,Exists
0,Current working directory,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True
1,Refactor root,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True
2,Target notebook,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True
3,Data reports folder,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True
4,Figures folder,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True


## 3. Helper Functions

In [3]:
def load_csv(filename: str, required: bool = True) -> pd.DataFrame:
    path = DATA_REPORTS_DIR / filename
    if not path.exists():
        if required:
            raise FileNotFoundError(f'Missing required input CSV: {path}')
        print(f'Optional CSV not available: {filename}')
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'Loaded {filename}: shape={df.shape}')
    return df


def save_table(df: pd.DataFrame, filename: str, title: str) -> Path:
    if not filename.startswith(REPORT_PREFIXES):
        raise ValueError(f'Report export notebook may only create report-prefixed files: {filename}')
    path = DATA_REPORTS_DIR / filename
    df.to_csv(path, index=False)
    print(title)
    display(df)
    print(f'Table shape: {df.shape}')
    print(f'Saved table: {path}')
    return path


def save_text(content: str, filename: str, title: str) -> Path:
    if not filename.startswith(REPORT_PREFIXES):
        raise ValueError(f'Report export notebook may only create report-prefixed files: {filename}')
    path = DATA_REPORTS_DIR / filename
    path.write_text(content, encoding='utf-8')
    print(title)
    print(f'Saved text file: {path}')
    return path


def first_value(df: pd.DataFrame, column: str, default: str = 'not available from current output'):
    if df.empty or column not in df.columns:
        return default
    value = df.iloc[0][column]
    if pd.isna(value):
        return default
    return value


def pct_text(value) -> str:
    if value == 'not available from current output' or pd.isna(value):
        return 'not available from current output'
    return f'{float(value):.2f}%'


def metric_text(value) -> str:
    if value == 'not available from current output' or pd.isna(value):
        return 'not available from current output'
    return f'{float(value):.4f}'


def require_models(df: pd.DataFrame, table_name: str) -> None:
    models = df['Model'].astype(str).tolist() if 'Model' in df.columns else []
    if models != MODEL_ORDER:
        raise ValueError(f'{table_name} must contain models in order {MODEL_ORDER}, got {models}')


def classify_driver_group(feature_name: str, current_group: str = '') -> str:
    text = f'{feature_name} {current_group}'.lower()
    if 'contract' in text:
        return 'Contract Type'
    if 'tenure' in text:
        return 'Tenure'
    if 'monthly' in text or 'total charges' in text or 'charges' in text:
        return 'Monthly Charges / Total Charges'
    if 'internet' in text or 'fiber' in text:
        return 'Internet Service'
    if 'dependents' in text:
        return 'Dependents'
    if 'tech support' in text or 'online security' in text or 'security' in text:
        return 'Tech Support / Online Security'
    if 'paperless' in text:
        return 'Paperless Billing'
    if 'service diversity' in text:
        return 'Service Diversity'
    if 'payment' in text or 'electronic check' in text:
        return 'Payment Method'
    return current_group or 'Other'


def get_best_model(best_df: pd.DataFrame, criterion: str) -> str:
    match = best_df[best_df['Criterion'] == criterion]
    if match.empty:
        return 'not available from current output'
    return str(match.iloc[0]['Model'])

## 4. Existing Output Discovery

In [4]:
all_csv_files = sorted(DATA_REPORTS_DIR.glob('*.csv'))
all_figure_files = sorted(FIGURES_DIR.glob('*.png'))
eda_csv_files = [path for path in all_csv_files if path.name.startswith('eda_')]
report_output_files_before = [path.name for path in all_csv_files if path.name.startswith(REPORT_PREFIXES)]

discovery_summary = pd.DataFrame([
    {'Output_Type': 'CSV files', 'Count': len(all_csv_files)},
    {'Output_Type': 'EDA CSV files', 'Count': len(eda_csv_files)},
    {'Output_Type': 'Figure files', 'Count': len(all_figure_files)},
    {'Output_Type': 'Existing report-prefixed files before rerun', 'Count': len(report_output_files_before)},
])
display(discovery_summary)
display(pd.DataFrame({'EDA_CSV': [path.name for path in eda_csv_files]}))
if not eda_csv_files:
    raise FileNotFoundError('At least one EDA CSV output is required for report summary generation.')

,Output_Type,Count
0,CSV files,77
1,EDA CSV files,16
2,Figure files,58
3,Existing report-prefixed files before rerun,7


,EDA_CSV
0,eda_churn_rate_by_contract.csv
1,eda_churn_rate_by_dependents.csv
2,eda_churn_rate_by_internet_service.csv
3,eda_churn_rate_by_online_security.csv
4,eda_churn_rate_by_paperless_billing.csv
5,eda_churn_rate_by_partner.csv
6,eda_churn_rate_by_payment_method.csv
7,eda_churn_rate_by_senior_citizen.csv
8,eda_churn_rate_by_service_diversity.csv
9,eda_churn_rate_by_tech_support.csv


## 5. Load EDA Outputs

In [5]:
eda_dataset_overview = load_csv('eda_dataset_overview.csv', required=False)
eda_numeric_summary = load_csv('eda_numeric_summary.csv', required=False)
eda_top_risk_segments = load_csv('eda_top_risk_segments.csv', required=False)
eda_churn_rate_by_contract = load_csv('eda_churn_rate_by_contract.csv', required=False)
eda_churn_rate_by_tenure_group = load_csv('eda_churn_rate_by_tenure_group.csv', required=False)
eda_churn_rate_by_internet_service = load_csv('eda_churn_rate_by_internet_service.csv', required=False)
eda_churn_rate_by_payment_method = load_csv('eda_churn_rate_by_payment_method.csv', required=False)
eda_churn_rate_by_tech_support = load_csv('eda_churn_rate_by_tech_support.csv', required=False)
eda_churn_rate_by_online_security = load_csv('eda_churn_rate_by_online_security.csv', required=False)
eda_churn_rate_by_paperless_billing = load_csv('eda_churn_rate_by_paperless_billing.csv', required=False)
eda_churn_rate_by_service_diversity = load_csv('eda_churn_rate_by_service_diversity.csv', required=False)

if eda_dataset_overview.empty and eda_numeric_summary.empty and eda_top_risk_segments.empty:
    raise ValueError('EDA outputs are present, but key report EDA tables are unavailable.')

Loaded eda_dataset_overview.csv: shape=(7, 2)
Loaded eda_numeric_summary.csv: shape=(15, 9)
Loaded eda_top_risk_segments.csv: shape=(27, 9)
Loaded eda_churn_rate_by_contract.csv: shape=(3, 8)
Loaded eda_churn_rate_by_tenure_group.csv: shape=(6, 8)
Loaded eda_churn_rate_by_internet_service.csv: shape=(3, 8)
Loaded eda_churn_rate_by_payment_method.csv: shape=(4, 8)
Loaded eda_churn_rate_by_tech_support.csv: shape=(2, 8)
Loaded eda_churn_rate_by_online_security.csv: shape=(2, 8)
Loaded eda_churn_rate_by_paperless_billing.csv: shape=(2, 8)
Loaded eda_churn_rate_by_service_diversity.csv: shape=(5, 8)


## 6. Load Model Comparison Outputs

In [6]:
default_validation_comparison = load_csv('core_model_default_threshold_validation_comparison.csv')
selected_validation_comparison = load_csv('core_model_selected_threshold_validation_comparison.csv')
selected_test_comparison = load_csv('core_model_selected_threshold_test_comparison.csv')
threshold_sensitivity_combined = load_csv('core_model_threshold_sensitivity_combined.csv')
best_model_summary = load_csv('core_model_best_model_summary.csv')
metric_ranking = load_csv('core_model_metric_ranking.csv')

require_models(selected_test_comparison, 'selected test comparison')
require_models(selected_validation_comparison, 'selected validation comparison')
require_models(default_validation_comparison, 'default validation comparison')

Loaded core_model_default_threshold_validation_comparison.csv: shape=(3, 17)
Loaded core_model_selected_threshold_validation_comparison.csv: shape=(3, 17)
Loaded core_model_selected_threshold_test_comparison.csv: shape=(3, 17)
Loaded core_model_threshold_sensitivity_combined.csv: shape=(273, 17)
Loaded core_model_best_model_summary.csv: shape=(6, 4)
Loaded core_model_metric_ranking.csv: shape=(18, 4)


## 7. Load Model Interpretation Outputs

In [7]:
logistic_interpretation = load_csv('model_interpretation_logistic_coefficients_summary.csv')
rf_interpretation = load_csv('model_interpretation_random_forest_top_features.csv')
xgb_gain_interpretation = load_csv('model_interpretation_xgboost_top_features.csv')
xgb_shap_interpretation = load_csv('model_interpretation_xgboost_shap_mean_abs.csv')
top_business_drivers = load_csv('model_interpretation_top_business_drivers.csv')
feature_mapping_used = load_csv('model_interpretation_feature_mapping_used.csv')

if top_business_drivers.empty:
    raise ValueError('Top business drivers table is required for Notebook 07.')

Loaded model_interpretation_logistic_coefficients_summary.csv: shape=(26, 7)
Loaded model_interpretation_random_forest_top_features.csv: shape=(15, 5)
Loaded model_interpretation_xgboost_top_features.csv: shape=(15, 5)
Loaded model_interpretation_xgboost_shap_mean_abs.csv: shape=(18, 6)
Loaded model_interpretation_top_business_drivers.csv: shape=(40, 8)
Loaded model_interpretation_feature_mapping_used.csv: shape=(18, 10)


## 8. Build Report-Ready EDA Summary

In [8]:
def top_segment(df: pd.DataFrame) -> pd.Series | None:
    if df.empty or 'churn_rate_pct' not in df.columns:
        return None
    return df.sort_values('churn_rate_pct', ascending=False).iloc[0]


rows_metric = first_value(eda_dataset_overview[eda_dataset_overview['metric'] == 'rows'], 'value') if not eda_dataset_overview.empty else 'not available from current output'
columns_metric = first_value(eda_dataset_overview[eda_dataset_overview['metric'] == 'columns'], 'value') if not eda_dataset_overview.empty else 'not available from current output'
churn_mean_df = eda_numeric_summary[eda_numeric_summary['column'] == 'Churn'] if not eda_numeric_summary.empty else pd.DataFrame()
overall_churn_rate = float(first_value(churn_mean_df, 'mean', 0)) * 100 if not churn_mean_df.empty else 'not available from current output'

eda_summary_rows = [
    {
        'Insight_Title': 'Dataset size',
        'Metric_Value': f'{rows_metric} rows, {columns_metric} columns',
        'Segment_or_Feature': 'Full dataset',
        'Short_Interpretation': 'Dataset size available from EDA overview.'
    },
    {
        'Insight_Title': 'Overall churn rate',
        'Metric_Value': pct_text(overall_churn_rate),
        'Segment_or_Feature': 'Churn',
        'Short_Interpretation': 'Overall churn rate used as context for segment-level risk.'
    },
]

for title, table in [
    ('Highest churn contract segment', eda_churn_rate_by_contract),
    ('Highest churn tenure segment', eda_churn_rate_by_tenure_group),
    ('Highest churn internet-service segment', eda_churn_rate_by_internet_service),
    ('Highest churn payment-method segment', eda_churn_rate_by_payment_method),
    ('Highest churn tech-support segment', eda_churn_rate_by_tech_support),
    ('Highest churn online-security segment', eda_churn_rate_by_online_security),
    ('Highest churn paperless-billing segment', eda_churn_rate_by_paperless_billing),
    ('Highest churn service-diversity segment', eda_churn_rate_by_service_diversity),
]:
    row = top_segment(table)
    if row is None:
        eda_summary_rows.append({
            'Insight_Title': title,
            'Metric_Value': 'not available from current output',
            'Segment_or_Feature': 'not available from current output',
            'Short_Interpretation': 'Source EDA table was not available.'
        })
    else:
        segment_label = row.get('category_display_name', row.get('segment_display_name', 'not available from current output'))
        feature_label = row.get('feature_display_name', row.get('feature', 'Segment'))
        eda_summary_rows.append({
            'Insight_Title': title,
            'Metric_Value': pct_text(row['churn_rate_pct']),
            'Segment_or_Feature': f'{feature_label} = {segment_label}',
            'Short_Interpretation': 'This segment had the highest churn rate in its EDA table.'
        })

if not eda_top_risk_segments.empty:
    top_segments_text = '; '.join(eda_top_risk_segments.head(3)['feature_segment_display'].astype(str).tolist())
else:
    top_segments_text = 'not available from current output'
eda_summary_rows.append({
    'Insight_Title': 'Top combined EDA risk segments',
    'Metric_Value': top_segments_text,
    'Segment_or_Feature': 'Top risk segments',
    'Short_Interpretation': 'Top EDA risk segments for report discussion.'
})

report_summary_eda = pd.DataFrame(eda_summary_rows)
save_table(report_summary_eda, 'report_summary_eda_key_insights.csv', 'Report-ready EDA key insights')

Report-ready EDA key insights


,Insight_Title,Metric_Value,Segment_or_Feature,Short_Interpretation
0,Dataset size,"7043 rows, 20 columns",Full dataset,Dataset size available from EDA overview.
1,Overall churn rate,26.54%,Churn,Overall churn rate used as context for segment...
2,Highest churn contract segment,42.71%,Contract Type = Month-to-month,This segment had the highest churn rate in its...
3,Highest churn tenure segment,47.44%,Tenure Group = 0-12 months,This segment had the highest churn rate in its...
4,Highest churn internet-service segment,41.89%,Internet Service = Fiber optic,This segment had the highest churn rate in its...
5,Highest churn payment-method segment,45.29%,Payment Method = Electronic check,This segment had the highest churn rate in its...
6,Highest churn tech-support segment,31.19%,Tech Support = No,This segment had the highest churn rate in its...
7,Highest churn online-security segment,31.33%,Online Security = No,This segment had the highest churn rate in its...
8,Highest churn paperless-billing segment,33.57%,Paperless Billing = Yes,This segment had the highest churn rate in its...
9,Highest churn service-diversity segment,38.85%,Service Diversity Score = 1,This segment had the highest churn rate in its...


Table shape: (11, 4)
Saved table: D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_eda_key_insights.csv


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_eda_key_insights.csv')

## 9. Build Report-Ready Model Comparison Summary

In [9]:
model_comparison_columns = ['Model', 'Threshold', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'PR_AUC', 'TP', 'FP', 'TN', 'FN']
report_summary_model_comparison = selected_test_comparison[model_comparison_columns].copy()
save_table(
    report_summary_model_comparison,
    'report_summary_model_comparison.csv',
    'Report-ready final selected-threshold model comparison',
)

Report-ready final selected-threshold model comparison


,Model,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TP,FP,TN,FN
0,Logistic Regression,0.43,0.729423,0.494845,0.854093,0.626632,0.850811,0.663918,240,245,531,41
1,Random Forest,0.17,0.682119,0.448405,0.850534,0.587224,0.823009,0.616782,239,294,482,42
2,XGBoost,0.38,0.721854,0.486974,0.864769,0.623077,0.841160,0.653840,243,256,520,38


Table shape: (3, 12)
Saved table: D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_model_comparison.csv


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_model_comparison.csv')

## 10. Build Report-Ready Threshold Summary

In [10]:
threshold_rows = []
for model_name in MODEL_ORDER:
    val_row = selected_validation_comparison[selected_validation_comparison['Model'] == model_name].iloc[0]
    test_row = selected_test_comparison[selected_test_comparison['Model'] == model_name].iloc[0]
    threshold_rows.append({
        'Model': model_name,
        'Selected threshold': val_row['Threshold'],
        'Validation recall': val_row['Recall'],
        'Validation F1': val_row['F1'],
        'Test recall': test_row['Recall'],
        'Test F1': test_row['F1'],
        'Meets recall target': bool(test_row['Meets_Recall_Target']),
    })
report_summary_threshold = pd.DataFrame(threshold_rows)
save_table(
    report_summary_threshold,
    'report_summary_threshold_selection.csv',
    'Report-ready validation-selected threshold summary',
)

Report-ready validation-selected threshold summary


,Model,Selected threshold,Validation recall,Validation F1,Test recall,Test F1,Meets recall target
0,Logistic Regression,0.43,0.853571,0.633952,0.854093,0.626632,True
1,Random Forest,0.17,0.867857,0.610553,0.850534,0.587224,True
2,XGBoost,0.38,0.850000,0.626316,0.864769,0.623077,True


Table shape: (3, 7)
Saved table: D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_threshold_selection.csv


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_threshold_selection.csv')

## 11. Build Report-Ready Model Recommendation Summary

In [11]:
recommended_balanced = get_best_model(best_model_summary, 'Recommended_Balanced_Model')
recommended_recall = get_best_model(best_model_summary, 'Recommended_Recall_Focused_Model')
best_recall = get_best_model(best_model_summary, 'Best model by Recall')
best_f1 = get_best_model(best_model_summary, 'Best model by F1')
best_roc = get_best_model(best_model_summary, 'Best model by ROC_AUC')
best_pr = get_best_model(best_model_summary, 'Best model by PR_AUC')

report_summary_recommendation = pd.DataFrame([
    {
        'Recommendation_Item': 'Recommended balanced model',
        'Model': recommended_balanced,
        'Short_Justification': 'Selected from final test metrics as the best balance among recall-target models.'
    },
    {
        'Recommendation_Item': 'Recommended recall-focused model',
        'Model': recommended_recall,
        'Short_Justification': 'Selected from final test metrics for maximum churn capture.'
    },
    {'Recommendation_Item': 'Best Recall model', 'Model': best_recall, 'Short_Justification': 'Loaded from best model summary.'},
    {'Recommendation_Item': 'Best F1 model', 'Model': best_f1, 'Short_Justification': 'Loaded from best model summary.'},
    {'Recommendation_Item': 'Best ROC_AUC model', 'Model': best_roc, 'Short_Justification': 'Loaded from best model summary.'},
    {'Recommendation_Item': 'Best PR_AUC model', 'Model': best_pr, 'Short_Justification': 'Loaded from best model summary.'},
])
save_table(
    report_summary_recommendation,
    'report_summary_best_model_recommendation.csv',
    'Report-ready best model recommendation summary',
)

Report-ready best model recommendation summary


,Recommendation_Item,Model,Short_Justification
0,Recommended balanced model,Logistic Regression,Selected from final test metrics as the best b...
1,Recommended recall-focused model,XGBoost,Selected from final test metrics for maximum c...
2,Best Recall model,XGBoost,Loaded from best model summary.
3,Best F1 model,Logistic Regression,Loaded from best model summary.
4,Best ROC_AUC model,Logistic Regression,Loaded from best model summary.
5,Best PR_AUC model,Logistic Regression,Loaded from best model summary.


Table shape: (6, 3)
Saved table: D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_best_model_recommendation.csv


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_best_model_recommendation.csv')

## 12. Build Report-Ready Top Churn Drivers Summary

In [12]:
driver_work = top_business_drivers.copy()
driver_work['Report_Driver_Group'] = driver_work.apply(
    lambda row: classify_driver_group(row.get('Feature_Display_Name', ''), row.get('Driver_Group', '')),
    axis=1,
)
logistic_direction_lookup = {}
if not logistic_interpretation.empty:
    logistic_direction_lookup = dict(zip(logistic_interpretation['Feature_Display_Name'], logistic_interpretation['Direction']))

driver_rows = []
for group_name, group_df in driver_work.groupby('Report_Driver_Group'):
    supporting_features = '; '.join(group_df['Feature_Display_Name'].dropna().astype(str).drop_duplicates().head(6).tolist())
    supporting_methods = '; '.join(group_df['Method'].dropna().astype(str).drop_duplicates().tolist())
    directions = []
    for feature_name in group_df['Feature_Display_Name'].dropna().astype(str).drop_duplicates().head(4):
        direction = logistic_direction_lookup.get(feature_name)
        if direction:
            directions.append(f'{feature_name}: {direction}')
    direction_text = '; '.join(directions) if directions else 'effect direction not available from current output'
    driver_rows.append({
        'Driver group': group_name,
        'Supporting features': supporting_features,
        'Supporting method': supporting_methods,
        'Direction / effect if available': direction_text,
        'Business interpretation': 'This driver group appears in model interpretation outputs and should be discussed as a churn risk signal.'
    })

report_summary_top_drivers = pd.DataFrame(driver_rows).sort_values('Driver group').reset_index(drop=True)
save_table(
    report_summary_top_drivers,
    'report_summary_top_churn_drivers.csv',
    'Report-ready top churn drivers summary',
)

Report-ready top churn drivers summary


,Driver group,Supporting features,Supporting method,Direction / effect if available,Business interpretation
0,Contract Type,Contract Type = Two year; Contract Type = One ...,Logistic coefficients; Random Forest native im...,Contract Type = Two year: decreases churn prob...,This driver group appears in model interpretat...
1,Dependents,Dependents = Yes; Dependents,Logistic coefficients; Random Forest native im...,Dependents = Yes: decreases churn probability,This driver group appears in model interpretat...
2,Internet Service,Internet Service = No Internet Service; Intern...,Logistic coefficients; Random Forest native im...,Internet Service = No Internet Service: decrea...,This driver group appears in model interpretat...
3,Monthly Charges / Total Charges,Total Charges; Monthly Charges; Monthly Charge...,Logistic coefficients; Random Forest native im...,Total Charges: decreases churn probability; Mo...,This driver group appears in model interpretat...
4,Other customer or service profile,Partner,XGBoost SHAP mean absolute impact,effect direction not available from current ou...,This driver group appears in model interpretat...
5,Paperless Billing,Paperless Billing = Yes; Paperless Billing,Logistic coefficients; XGBoost gain importance...,Paperless Billing = Yes: increases churn proba...,This driver group appears in model interpretat...
6,Payment Method,Payment Method,Random Forest native importance; XGBoost SHAP ...,effect direction not available from current ou...,This driver group appears in model interpretat...
7,Service Diversity,Service Diversity Score,Random Forest native importance; XGBoost gain ...,Service Diversity Score: decreases churn proba...,This driver group appears in model interpretat...
8,Tech Support / Online Security,Tech Support; Online Security,XGBoost gain importance; XGBoost SHAP mean abs...,effect direction not available from current ou...,This driver group appears in model interpretat...
9,Tenure,Tenure; Tenure Group,Logistic coefficients; Random Forest native im...,Tenure: decreases churn probability,This driver group appears in model interpretat...


Table shape: (10, 5)
Saved table: D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_top_churn_drivers.csv


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_top_churn_drivers.csv')

## 13. Build Business Recommendations Table

In [13]:
def evidence_for(group_name: str) -> str:
    match = report_summary_top_drivers[report_summary_top_drivers['Driver group'] == group_name]
    if match.empty:
        return 'not available from current output'
    return match.iloc[0]['Supporting features']


business_recommendations = pd.DataFrame([
    {
        'Recommendation': 'Contract retention strategy',
        'Target segment': 'Month-to-month and contract-sensitive customers',
        'Supporting evidence': evidence_for('Contract Type'),
        'Expected business value': 'Reduce voluntary churn by encouraging longer commitments or targeted renewal offers.',
        'Related model/EDA evidence': 'Contract type appears in EDA and model interpretation outputs.'
    },
    {
        'Recommendation': 'Early-tenure customer support',
        'Target segment': 'Short-tenure customers',
        'Supporting evidence': evidence_for('Tenure'),
        'Expected business value': 'Improve onboarding and reduce early lifecycle churn.',
        'Related model/EDA evidence': 'Tenure appears in EDA risk segments and model interpretation outputs.'
    },
    {
        'Recommendation': 'High-charge customer monitoring',
        'Target segment': 'Customers with high monthly charges or charge-related risk signals',
        'Supporting evidence': evidence_for('Monthly Charges / Total Charges'),
        'Expected business value': 'Identify price-sensitive customers and target plan-fit interventions.',
        'Related model/EDA evidence': 'Charge-related variables appear across interpretation outputs.'
    },
    {
        'Recommendation': 'Fiber optic / internet-service issue review',
        'Target segment': 'Internet-service customers with elevated churn risk',
        'Supporting evidence': evidence_for('Internet Service'),
        'Expected business value': 'Reduce service-experience churn through issue review and service quality follow-up.',
        'Related model/EDA evidence': 'Internet service appears in EDA and XGBoost interpretation outputs.'
    },
    {
        'Recommendation': 'Tech support and online security bundling',
        'Target segment': 'Customers without support or security add-ons',
        'Supporting evidence': evidence_for('Tech Support / Online Security'),
        'Expected business value': 'Increase stickiness by bundling support and security services.',
        'Related model/EDA evidence': 'Support/security variables appear in EDA and interpretation outputs.'
    },
    {
        'Recommendation': 'Payment method and paperless billing monitoring',
        'Target segment': 'Customers with billing-behavior risk signals',
        'Supporting evidence': f"{evidence_for('Payment Method')}; {evidence_for('Paperless Billing')}",
        'Expected business value': 'Improve billing journey and prioritize outreach for billing-related churn signals.',
        'Related model/EDA evidence': 'Payment and paperless billing appear in EDA and model interpretation outputs.'
    },
])
save_table(
    business_recommendations,
    'report_summary_business_recommendations.csv',
    'Report-ready business recommendations',
)

Report-ready business recommendations


,Recommendation,Target segment,Supporting evidence,Expected business value,Related model/EDA evidence
0,Contract retention strategy,Month-to-month and contract-sensitive customers,Contract Type = Two year; Contract Type = One ...,Reduce voluntary churn by encouraging longer c...,Contract type appears in EDA and model interpr...
1,Early-tenure customer support,Short-tenure customers,Tenure; Tenure Group,Improve onboarding and reduce early lifecycle ...,Tenure appears in EDA risk segments and model ...
2,High-charge customer monitoring,Customers with high monthly charges or charge-...,Total Charges; Monthly Charges; Monthly Charge...,Identify price-sensitive customers and target ...,Charge-related variables appear across interpr...
3,Fiber optic / internet-service issue review,Internet-service customers with elevated churn...,Internet Service = No Internet Service; Intern...,Reduce service-experience churn through issue ...,Internet service appears in EDA and XGBoost in...
4,Tech support and online security bundling,Customers without support or security add-ons,Tech Support; Online Security,Increase stickiness by bundling support and se...,Support/security variables appear in EDA and i...
5,Payment method and paperless billing monitoring,Customers with billing-behavior risk signals,Payment Method; Paperless Billing = Yes; Paper...,Improve billing journey and prioritize outreac...,Payment and paperless billing appear in EDA an...


Table shape: (6, 5)
Saved table: D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_business_recommendations.csv


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_business_recommendations.csv')

## 14. Build Figure Manifest For Report

In [14]:
figure_manifest_specs = [
    ('eda_01_churn_distribution.png', 'Notebook 01 EDA', 'EDA overview', 'Overall churn distribution.'),
    ('eda_02_churn_rate_by_contract.png', 'Notebook 01 EDA', 'EDA segment analysis', 'Churn rate by contract type.'),
    ('eda_04_churn_rate_by_internet_service.png', 'Notebook 01 EDA', 'EDA segment analysis', 'Churn rate by internet service.'),
    ('eda_03_churn_rate_by_payment_method.png', 'Notebook 01 EDA', 'EDA segment analysis', 'Churn rate by payment method.'),
    ('eda_14_top_risk_segments.png', 'Notebook 01 EDA', 'EDA risk summary', 'Top high-risk churn segments.'),
    ('logistic_08_top_absolute_coefficients.png', 'Notebook 02 Logistic Regression', 'Model interpretation', 'Logistic Regression strongest coefficients.'),
    ('random_forest_06_top_feature_importance.png', 'Notebook 03 Random Forest', 'Model interpretation', 'Random Forest top native feature importance.'),
    ('xgboost_06_top_feature_importance.png', 'Notebook 04 XGBoost', 'Model interpretation', 'XGBoost top gain-based feature importance.'),
    ('model_comparison_01_selected_test_metrics_bar.png', 'Notebook 05 Model Comparison', 'Model comparison', 'Final selected-threshold test metric comparison.'),
    ('model_interpretation_04_xgboost_shap_summary_bar.png', 'Notebook 06 Model Interpretation', 'SHAP interpretation', 'XGBoost SHAP summary bar plot.'),
    ('model_interpretation_05_xgboost_shap_beeswarm.png', 'Notebook 06 Model Interpretation', 'SHAP interpretation', 'XGBoost SHAP beeswarm plot.'),
]
figure_manifest = pd.DataFrame([
    {
        'Figure filename': filename,
        'Notebook source': notebook_source,
        'Suggested report section': section,
        'Short caption': caption,
        'File exists': (FIGURES_DIR / filename).exists(),
        'Path': FIGURES_DIR / filename,
    }
    for filename, notebook_source, section, caption in figure_manifest_specs
])
save_table(figure_manifest, 'report_figure_manifest.csv', 'Report figure manifest')

Report figure manifest


,Figure filename,Notebook source,Suggested report section,Short caption,File exists,Path
0,eda_01_churn_distribution.png,Notebook 01 EDA,EDA overview,Overall churn distribution.,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
1,eda_02_churn_rate_by_contract.png,Notebook 01 EDA,EDA segment analysis,Churn rate by contract type.,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
2,eda_04_churn_rate_by_internet_service.png,Notebook 01 EDA,EDA segment analysis,Churn rate by internet service.,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
3,eda_03_churn_rate_by_payment_method.png,Notebook 01 EDA,EDA segment analysis,Churn rate by payment method.,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
4,eda_14_top_risk_segments.png,Notebook 01 EDA,EDA risk summary,Top high-risk churn segments.,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
5,logistic_08_top_absolute_coefficients.png,Notebook 02 Logistic Regression,Model interpretation,Logistic Regression strongest coefficients.,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
6,random_forest_06_top_feature_importance.png,Notebook 03 Random Forest,Model interpretation,Random Forest top native feature importance.,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
7,xgboost_06_top_feature_importance.png,Notebook 04 XGBoost,Model interpretation,XGBoost top gain-based feature importance.,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
8,model_comparison_01_selected_test_metrics_bar.png,Notebook 05 Model Comparison,Model comparison,Final selected-threshold test metric comparison.,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
9,model_interpretation_04_xgboost_shap_summary_b...,Notebook 06 Model Interpretation,SHAP interpretation,XGBoost SHAP summary bar plot.,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...


Table shape: (11, 6)
Saved table: D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_figure_manifest.csv


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_figure_manifest.csv')

## 15. Save Report-Ready CSV Outputs

In [15]:
expected_report_csvs = [
    'report_summary_eda_key_insights.csv',
    'report_summary_model_comparison.csv',
    'report_summary_threshold_selection.csv',
    'report_summary_best_model_recommendation.csv',
    'report_summary_top_churn_drivers.csv',
    'report_summary_business_recommendations.csv',
    'report_figure_manifest.csv',
]
report_csv_check = pd.DataFrame([
    {'File': filename, 'Path': DATA_REPORTS_DIR / filename, 'Created': (DATA_REPORTS_DIR / filename).exists()}
    for filename in expected_report_csvs
])
display(report_csv_check)
if not report_csv_check['Created'].all():
    missing = report_csv_check.loc[~report_csv_check['Created'], 'File'].tolist()
    raise FileNotFoundError(f'Missing required report summary outputs: {missing}')

,File,Path,Created
0,report_summary_eda_key_insights.csv,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True
1,report_summary_model_comparison.csv,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True
2,report_summary_threshold_selection.csv,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True
3,report_summary_best_model_recommendation.csv,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True
4,report_summary_top_churn_drivers.csv,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True
5,report_summary_business_recommendations.csv,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True
6,report_figure_manifest.csv,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...,True


## 16. Save Optional Markdown and LaTeX Tables

In [16]:
save_text(
    report_summary_model_comparison.to_latex(index=False, float_format='%.4f'),
    'report_table_model_comparison_latex.tex',
    'LaTeX model comparison table',
)
save_text(
    report_summary_threshold.to_latex(index=False, float_format='%.4f'),
    'report_table_threshold_selection_latex.tex',
    'LaTeX threshold selection table',
)
save_text(
    report_summary_top_drivers.to_latex(index=False),
    'report_table_top_churn_drivers_latex.tex',
    'LaTeX top churn drivers table',
)

markdown_summary = f"""# Report Summary

## Dataset / EDA Conclusion
Churn is meaningfully associated with contract type, tenure, charges, internet service, support services, and payment behavior. Overall churn rate from current EDA output: {pct_text(overall_churn_rate)}.

## Modeling Conclusion
Recommended balanced model: {recommended_balanced}. Recommended recall-focused model: {recommended_recall}. Random Forest provides useful feature-importance support but is not the top-performing model in the final comparison table.

## Threshold Conclusion
Validation-based threshold sensitivity was used in the modeling notebooks. The test set was used only for final evaluation. Final test recall values were: {', '.join(report_summary_threshold.apply(lambda row: f'{row['Model']}={row['Test recall']:.4f}', axis=1).tolist())}.

## Interpretation Conclusion
Main churn drivers include contract type, tenure, charges, internet service, dependents, support/security services, paperless billing, service diversity, and payment method. SHAP was used only for interpretation of the recreated XGBoost-style model and does not change model selection.

## Business Recommendation Conclusion
Focus retention on month-to-month, short-tenure, and high-charge customers; improve support and security service adoption; review internet service experience; and use the recall-focused model for risk screening with the balanced model for report discussion.
"""
save_text(markdown_summary, 'report_summary_markdown.md', 'Markdown report summary')

optional_output_check = pd.DataFrame([
    {'File': filename, 'Created': (DATA_REPORTS_DIR / filename).exists(), 'Path': DATA_REPORTS_DIR / filename}
    for filename in [
        'report_table_model_comparison_latex.tex',
        'report_table_threshold_selection_latex.tex',
        'report_table_top_churn_drivers_latex.tex',
        'report_summary_markdown.md',
    ]
])
display(optional_output_check)

SyntaxError: f-string: f-string: unmatched '[' (125389122.py, line 26)

## 17. Final Report Narrative Summary

In [ ]:
print('Report-ready narrative summary')
print('1. Dataset / EDA conclusion: churn is associated with contract type, tenure, charges, internet service, support services, and payment behavior.')
print(f'2. Modeling conclusion: {recommended_balanced} is the recommended balanced model; {recommended_recall} is the recommended recall-focused model.')
print('3. Threshold conclusion: validation-based threshold sensitivity was used, and test results were used for final evaluation only.')
print('4. Interpretation conclusion: main churn drivers include contract type, tenure, charges, internet service, dependents, and support/security services.')
print('5. Business recommendation conclusion: focus on contract retention, early-tenure support, high-charge monitoring, internet-service review, support/security bundling, and billing-behavior monitoring.')

## 18. Source Safety Confirmation

In [ ]:
notebook_source = TARGET_NOTEBOOK_PATH.read_text(encoding='utf-8')
forbidden_tokens = [
    'run' + '_' + 'pipeline',
    'Logistic' + 'Regression(',
    'Random' + 'Forest' + 'Classifier',
    'XGB' + 'Classifier',
    'import ' + 'shap',
    '.fi' + 't(',
    'predict' + '_proba',
]
found_forbidden_tokens = [token for token in forbidden_tokens if token in notebook_source]
if found_forbidden_tokens:
    raise ValueError(f'Forbidden execution/training tokens found in Notebook 07 source: {found_forbidden_tokens}')

print('Notebook 07 source safety confirmation')
print('No preprocessing orchestration call is present.')
print('No model classes are imported.')
print('No model training call or probability generation call is present.')
print('No SHAP package import or recomputation is present.')
print('Notebook writes only report-prefixed CSV, Markdown, and LaTeX summary files.')
print('No source files, checkpoint files, existing notebooks, model files, SQL files, or legacy files are modified by notebook code.')